# Chapter 5: Cross-Condition Analysis (v4-definitive)
## Three-Path Strategy: Multi-Session Averaging + Method Comparison + Circuit-Specific Analysis

---

**Kernel:** R  |  **Cores:** max − 2  |  **Est. runtime:** ~18 h

---

### Motivation

v3-final established three critical facts:
1. Per-subject R_SCALE produces a **null** HC-MDD comparison (d = −0.12, p = 0.70)
2. The bifurcation parameter has **poor ICC** (HC: 0.06, MDD: 0.14) — single sessions are noisy
3. The parameter value is **estimation-method-dependent** (UKF ≈ −0.25, spectral ≈ −0.15, FC-match ≈ +0.05)

This notebook addresses all three issues with a combined strategy:

### Path 1 — Multi-Session Averaging
HNU1 provides 10 sessions per HC subject. Averaging $a$ across sessions reduces noise
by $\sqrt{10} \approx 3.2\times$, producing reliable trait-level estimates. MDD rest1+rest2
averaging provides $\sqrt{2}$ improvement. With reliable estimates, TOST equivalence
testing becomes achievable.

### Path 2 — Estimation Method Comparison
Systematic comparison of three estimation methods on the same data:
- UKF state-space (per-subject R_SCALE)
- Spectral Lorentzian NLS fitting
- FC-matching simulation

This is the primary **methodological** contribution.

### Path 3 — Depression-Circuit Analysis
Instead of whole-brain averages, test specific ROIs implicated in MDD:
amygdala, hippocampus, mPFC, dACC, subgenual cingulate, dlPFC.
Network-level averages can dilute focal effects in circuits known to be
dysfunctional in depression.

---

# 1 — Libraries and Configuration

In [1]:
suppressPackageStartupMessages({
  library(pracma); library(MASS); library(Matrix)
  library(dplyr); library(tidyr); library(ggplot2)
  library(scales); library(parallel); library(zoo)
})
source("R/constants.R"); source("R/ukf_engine.R"); source("R/optim.R")
source("R/preprocessing.R"); source("R/sl_models.R")

# Paths
MDD_216_R1 <- "data/parcellated/219roi/rest1"
MDD_216_R2 <- "data/parcellated/219roi/rest2"
HC_216_DIR <- "data/parcellated_hc/216roi"
RESULTS_DIR <- "results/ch5_v4def"
LOG_FILE <- file.path(RESULTS_DIR, "ch5_v4def_log.txt")
dir.create(RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)

TR <- 2.0; DT <- TR * UKF_CONSTANTS$DT_FRACTION
MAXSTEPS_S1 <- 200L; CHISQ_OBS_FLOOR <- 0.638
CHISQ_TOL_S1 <- CHISQ_OBS_FLOOR * 1e-4
A_MIN <- -2.0; A_MAX <- 2.0
N_CORES <- max(1L, detectCores() - 2L)

YEO_NETWORKS <- c(Vis="Visual",SomMot="Somatomotor",DorsAttn="Dorsal Attention",
  SalVentAttn="Salience/VentAttn",Limbic="Limbic",Cont="Frontoparietal",Default="Default Mode")
parse_network <- function(roi_name) {
  for (p in names(YEO_NETWORKS)) if (grepl(paste0("^7Networks_[LR]H_",p), roi_name)) return(YEO_NETWORKS[[p]])
  "Subcortical"
}

# Depression-relevant ROIs (Path 3)
# Subcortical targets from Melbourne atlas
MDD_CIRCUIT_SUBCORT <- c("Amyg-lh","Amyg-rh","Hipp-lh","Hipp-rh",
                          "NAcc-lh","NAcc-rh","Thal-lh","Thal-rh")
# Cortical targets from Schaefer — mPFC, dACC, dlPFC, subgenual, OFC
MDD_CIRCUIT_CORTICAL_PATTERNS <- c(
  "Default_PFC",      # medial PFC (DMN prefrontal)
  "Cont_PFCl",        # dorsolateral PFC (frontoparietal)
  "SalVentAttn_Med",  # dorsal ACC / salience medial
  "Default_Temp",     # temporal pole / MTL (DMN temporal)
  "Limbic_OFC",       # orbitofrontal cortex
  "Limbic_TempPole"   # temporal pole limbic
)

cat(sprintf("Configuration loaded. Cores: %d\n", N_CORES))

Warning message:
“package ‘pracma’ was built under R version 4.3.3”
Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”
Warning message:
“package ‘scales’ was built under R version 4.3.3”
Warning message:
“package ‘zoo’ was built under R version 4.3.3”
KernSmooth 2.23 loaded
Copyright M. P. Wand 1997-2009



Configuration loaded. Cores: 16


# 2 — Logging

In [2]:
log_con <- file(LOG_FILE, open = "w")
log_msg <- function(msg) {
  ts <- sprintf("[%s] %s", format(Sys.time(), "%H:%M:%S"), msg)
  cat(ts, "\n"); writeLines(ts, log_con); flush(log_con)
}
log_section <- function(t) {
  b <- paste(rep("=",70),collapse="")
  for(x in c("",b,t,b)){writeLines(x,log_con)}; flush(log_con)
  cat("\n",b,"\n",t,"\n",b,"\n",sep="")
}
`%||%` <- function(a,b) if(!is.null(a)) a else b

log_section(sprintf("CH5 v4-DEFINITIVE — %s", Sys.time()))


CH5 v4-DEFINITIVE — 2026-03-27 05:57:38.429183


# 3 — Data Loading

Load **all available sessions**: HC has up to 10 sessions per subject in HNU1,
MDD has 2 (rest1 pre-NF, rest2 post-NF).

In [3]:
log_section("DATA LOADING")

load_csvs <- function(folder, tag="") {
  files <- list.files(folder, "\\.csv$", full.names=TRUE)
  if(!length(files)){warning(paste("No CSVs:",folder));return(list())}
  out <- list()
  for(fp in sort(files)){
    base <- tools::file_path_sans_ext(basename(fp))
    df <- read.csv(fp, check.names=FALSE)
    drop <- which(tolower(names(df)) %in% c("time","tr","t","") |
                  grepl("(?i)^background$|^0$",names(df),perl=TRUE))
    if(length(drop)>0) df <- df[,-drop,drop=FALSE]
    out[[base]] <- df
  }
  log_msg(sprintf("  %s: %d files", tag, length(out))); out
}

# MDD
mdd_r1_raw <- load_csvs(MDD_216_R1, "MDD r1")
mdd_r2_raw <- load_csvs(MDD_216_R2, "MDD r2")

# Clean MDD names: strip _rest1_219roi etc
clean_mdd <- function(lst) {
  nms <- names(lst)
  nms <- sub("_(rest[12])_219roi$", "", nms)
  nms <- sub("_219roi$", "", nms)
  setNames(lst, nms)
}
mdd_r1_raw <- clean_mdd(mdd_r1_raw); mdd_r2_raw <- clean_mdd(mdd_r2_raw)
mdd_subs <- intersect(names(mdd_r1_raw), names(mdd_r2_raw))

# HC — load all, split by session number
hc_all_raw <- load_csvs(HC_216_DIR, "HC all")

# Parse HC filenames: 0025427_session_1_216roi → subject=0025427, session=1
hc_parsed <- data.frame(
  key = names(hc_all_raw),
  subject = sub("_session_\\d+.*$", "", names(hc_all_raw)),
  session = as.integer(sub(".*session_(\\d+).*", "\\1", names(hc_all_raw))),
  stringsAsFactors = FALSE
)
hc_subs <- sort(unique(hc_parsed$subject))
hc_sessions <- sort(unique(hc_parsed$session))

# Organise into nested list: hc_by_sess[[session_num]][[subject_id]] = df
hc_by_sess <- list()
for (s in hc_sessions) {
  rows <- hc_parsed[hc_parsed$session == s, ]
  hc_by_sess[[as.character(s)]] <- setNames(
    hc_all_raw[rows$key], rows$subject
  )
}

roi_cols <- intersect(colnames(mdd_r1_raw[[1]]), colnames(hc_by_sess[["1"]][[1]]))
N_ROIS <- length(roi_cols)

log_msg(sprintf("MDD: %d subjects × 2 sessions", length(mdd_subs)))
log_msg(sprintf("HC:  %d subjects × %d sessions", length(hc_subs), length(hc_sessions)))
log_msg(sprintf("ROIs: %d", N_ROIS))


DATA LOADING
[05:57:38]   MDD r1: 20 files 
[05:57:39]   MDD r2: 20 files 
[05:57:50]   HC all: 295 files 
[05:57:51] MDD: 19 subjects × 2 sessions 
[05:57:51] HC:  30 subjects × 10 sessions 
[05:57:51] ROIs: 216 


# 4 — Smoothing (All Sessions)

In [4]:
log_section("SMOOTHING")

interp_na <- function(x) {
  if (!anyNA(x)) return(x)
  x <- approx(seq_along(x), x, seq_along(x), rule = 2)$y
  x <- zoo::na.locf(x, na.rm = FALSE)
  x <- zoo::na.locf(x, fromLast = TRUE, na.rm = FALSE)
  if (anyNA(x)) x[is.na(x)] <- 0; x
}

smooth_list <- function(data_list, subjects, preset = "light") {
  lapply(subjects, function(sid) {
    sm <- smooth_subject_data(data_list[[sid]], preset = preset, verbose = FALSE)
    df <- as.data.frame(sm$smoothed)
    fn <- tolower(names(df)[1])
    is_t <- grepl("^(time|tr|t)$", fn)
    if (!is_t && nrow(df) > 1 && all(diff(df[[1]]) > 0) &&
        abs(mean(diff(df[[1]])) - TR) < TR * 0.1) is_t <- TRUE
    if (!is_t) df <- cbind(Time = seq(0, (nrow(df)-1)*TR, by = TR), df)
    for (col in names(df)[-1]) df[[col]] <- interp_na(df[[col]])
    df
  }) |> setNames(subjects)
}

# MDD
cat("Smoothing MDD...\n")
sm_mdd_r1 <- smooth_list(mdd_r1_raw, mdd_subs)
sm_mdd_r2 <- smooth_list(mdd_r2_raw, mdd_subs)

# HC — all sessions
sm_hc <- list()
for (s in hc_sessions) {
  cat(sprintf("Smoothing HC session %d...\n", s))
  subs_in_sess <- intersect(hc_subs, names(hc_by_sess[[as.character(s)]]))
  sm_hc[[as.character(s)]] <- smooth_list(hc_by_sess[[as.character(s)]], subs_in_sess)
}

# Verify ROI overlap
roi_cols <- intersect(colnames(sm_mdd_r1[[1]])[-1], colnames(sm_hc[["1"]][[1]])[-1])
N_ROIS <- length(roi_cols)
log_msg(sprintf("Smoothing complete. ROIs: %d  HC sessions: %d", N_ROIS, length(hc_sessions)))


SMOOTHING
Smoothing MDD...


Warning message in select_bandwidth(signal = sig, preset = preset, TR = TR, verbose = verbose, :
“[7Networks_LH_Limbic_TempPole_4] Low variance retention: 29.5% < threshold 30%. Consider using 'light' preset.”
Warning message in select_bandwidth(signal = sig, preset = preset, TR = TR, verbose = verbose, :
“[7Networks_LH_Limbic_TempPole_4] Low variance retention: 27.2% < threshold 30%. Consider using 'light' preset.”


Smoothing HC session 1...
Smoothing HC session 2...
Smoothing HC session 3...
Smoothing HC session 4...
Smoothing HC session 5...
Smoothing HC session 6...
Smoothing HC session 7...
Smoothing HC session 8...
Smoothing HC session 9...
Smoothing HC session 10...
[05:58:03] Smoothing complete. ROIs: 216  HC sessions: 10 


# 5 — Per-Subject R_SCALE Calibration

In [5]:
log_section("PER-SUBJECT R_SCALE CALIBRATION")

calibrate_rscale <- function(smoothed_df, rois, n_pilot = 20) {
  pilot <- rois[seq(1, length(rois), length.out = min(n_pilot, length(rois)))]
  sds <- c()
  for (roi in pilot) {
    ts <- smoothed_df[[roi]]
    if (is.null(ts) || all(is.na(ts))) next
    if (anyNA(ts)) ts <- interp_na(ts)
    ts <- ts - mean(ts, na.rm = TRUE)
    ha <- hilbert_analytic(ts)
    sc <- if (!is.null(ha$amp_scale) && is.finite(ha$amp_scale)) ha$amp_scale else 1
    sds <- c(sds, sd(c(ha$real * sc, ha$imag * sc), na.rm = TRUE))
  }
  if (length(sds) == 0) return(0.15)
  median(sds, na.rm = TRUE)
}

# HC — calibrate from session 1
rscale_hc <- data.frame(subject = hc_subs, r_scale = NA_real_)
for (i in seq_along(hc_subs)) {
  rscale_hc$r_scale[i] <- calibrate_rscale(sm_hc[["1"]][[hc_subs[i]]], roi_cols)
}

# MDD — calibrate from rest1
rscale_mdd <- data.frame(subject = mdd_subs, r_scale = NA_real_)
for (i in seq_along(mdd_subs)) {
  rscale_mdd$r_scale[i] <- calibrate_rscale(sm_mdd_r1[[mdd_subs[i]]], roi_cols)
}

log_msg(sprintf("HC  R_SCALE: %.4f ± %.4f", mean(rscale_hc$r_scale), sd(rscale_hc$r_scale)))
log_msg(sprintf("MDD R_SCALE: %.4f ± %.4f", mean(rscale_mdd$r_scale), sd(rscale_mdd$r_scale)))
t_rs <- t.test(rscale_hc$r_scale, rscale_mdd$r_scale)
log_msg(sprintf("Group diff: t=%.3f p=%.4f", t_rs$statistic, t_rs$p.value))


PER-SUBJECT R_SCALE CALIBRATION
[05:58:03] HC  R_SCALE: 0.2169 ± 0.0301 
[05:58:03] MDD R_SCALE: 0.2164 ± 0.0570 
[05:58:03] Group diff: t=0.041 p=0.9677 


# 6 — Core UKF Functions

Uses the **proven named-argument pattern** from v1/v2 that produced 100% convergence.

In [6]:
omega_from_phase_v2 <- function(ha, TR = 2.0) {
  unwrap <- function(ph) { d <- diff(ph); d <- d - 2*pi*round(d/(2*pi)); c(ph[1], ph[1]+cumsum(d)) }
  dph <- diff(unwrap(ha$phase))
  dph_c <- dph[dph >= quantile(dph, 0.10, na.rm=TRUE) & dph <= quantile(dph, 0.90, na.rm=TRUE)]
  om <- median(dph_c, na.rm = TRUE)
  if (is.na(om) || !is.finite(om) || om <= 0) om <- SL_BOUNDS$OM_MIN
  max(SL_BOUNDS$OM_MIN, min(SL_BOUNDS$OM_MAX, om))
}

run_sl_fit <- function(smoothed_df, roi_name, sid, session, r_scale) {
  q_scale <- sqrt(r_scale * 0.001)
  tryCatch({
    ts_mat <- prepare_sl_single_input(smoothed_df, roi_name)
    TRIM <- 5L
    if (nrow(ts_mat) > 2 * TRIM + 20) {
      idx <- (TRIM + 1):(nrow(ts_mat) - TRIM)
      ts_mat <- ts_mat[idx, , drop = FALSE]
      ts_mat[, 1] <- seq(0, (nrow(ts_mat) - 1) * TR, by = TR)
    }
    x_bold <- ts_mat[, 2]
    if (all(is.na(x_bold)) || sd(x_bold, na.rm = TRUE) < 1e-6 || !any(is.finite(x_bold)))
      return(list(subject=sid, session=session, roi=roi_name, a=NA_real_,
                  omega_hz=NA_real_, chisq=NA_real_, r_scale=r_scale))
    roi_ts <- smoothed_df[[roi_name]]
    if (anyNA(roi_ts)) roi_ts <- interp_na(roi_ts)
    ha <- hilbert_analytic(roi_ts)
    om_init <- omega_from_phase_v2(ha, TR = TR)
    model <- make_sl_single_fixed_om(om_init)
    result <- iterative_param_optim(
      param_guess = c(SL_INIT$A_INIT), t_dummy = 0, ts_data = ts_mat,
      ode_model   = model, N_p = 1L, N_y = 2L, dt = DT, dT = TR,
      param_tol   = 1e-4, MAXSTEPS = MAXSTEPS_S1,
      R_scale     = r_scale, Q_scale = q_scale,
      forcePositive = FALSE, seeded = FALSE,
      param_lower = c(A_MIN), param_upper = c(A_MAX),
      chisq_plateau_tol = CHISQ_TOL_S1)
    list(subject=sid, session=session, roi=roi_name, a=result$par[1],
         omega_hz=om_init/(2*pi*TR), chisq=result$value, r_scale=r_scale)
  }, error = function(e) {
    list(subject=sid, session=session, roi=roi_name, a=NA_real_,
         omega_hz=NA_real_, chisq=NA_real_, r_scale=r_scale,
         err=conditionMessage(e))
  })
}

# ── Diagnostic (naked, no tryCatch — errors are visible) ────────
cat("Sequential diagnostic...\n")
d_sid <- hc_subs[1]; d_roi <- roi_cols[1]; d_rs <- rscale_hc$r_scale[1]
d_df <- sm_hc[["1"]][[d_sid]]
d_ts <- prepare_sl_single_input(d_df, d_roi)
d_rts <- d_df[[d_roi]]; if(anyNA(d_rts)) d_rts <- interp_na(d_rts)
d_ha <- hilbert_analytic(d_rts); d_om <- omega_from_phase_v2(d_ha)
d_mod <- make_sl_single_fixed_om(d_om); d_qs <- sqrt(d_rs * 0.001)
d_res <- iterative_param_optim(
  param_guess=c(SL_INIT$A_INIT), t_dummy=0, ts_data=d_ts,
  ode_model=d_mod, N_p=1L, N_y=2L, dt=DT, dT=TR,
  param_tol=1e-4, MAXSTEPS=MAXSTEPS_S1,
  R_scale=d_rs, Q_scale=d_qs,
  forcePositive=FALSE, seeded=FALSE,
  param_lower=c(A_MIN), param_upper=c(A_MAX),
  chisq_plateau_tol=CHISQ_TOL_S1)
cat(sprintf("  DIAGNOSTIC OK: a=%.4f chisq=%.4f\n", d_res$par[1], d_res$value))

cat("UKF functions defined and verified.\n")

Sequential diagnostic...
  DIAGNOSTIC OK: a=-0.1424 chisq=0.6447
UKF functions defined and verified.


# 7 — Parallel Batch Engine

Robust parallel execution with automatic sequential fallback.

In [7]:
run_batch <- function(smooth_list, jobs, label) {
  cat(sprintf("  %s: %d fits (%d cores)\n", label, nrow(jobs), N_CORES))
  wd <- getwd()
  cl <- makeCluster(N_CORES, type = "PSOCK")
  on.exit(stopCluster(cl), add = TRUE)
  clusterExport(cl, c("jobs", "smooth_list", "wd"), envir = environment())
  clusterExport(cl, c("run_sl_fit", "interp_na", "omega_from_phase_v2",
    "TR", "DT", "MAXSTEPS_S1", "CHISQ_TOL_S1", "A_MIN", "A_MAX"),
    envir = globalenv())
  init_ok <- tryCatch({
    clusterEvalQ(cl, {
      setwd(wd)
      suppressPackageStartupMessages({library(pracma);library(MASS);library(Matrix);library(zoo)})
      source(file.path(wd,"R","constants.R")); source(file.path(wd,"R","ukf_engine.R"))
      source(file.path(wd,"R","optim.R")); source(file.path(wd,"R","preprocessing.R"))
      source(file.path(wd,"R","sl_models.R"))
      `%||%` <- function(a,b) if(!is.null(a)) a else b; TRUE
    }); TRUE
  }, error = function(e) { cat(sprintf("  INIT FAIL: %s\n", conditionMessage(e))); FALSE })
  
  t0 <- proc.time()
  if (init_ok) {
    results <- tryCatch(
      parLapply(cl, seq_len(nrow(jobs)), function(i)
        run_sl_fit(smooth_list[[jobs$sid[i]]], jobs$roi[i],
                   jobs$sid[i], jobs$session[i], jobs$r_scale[i])),
      error = function(e) { cat(sprintf("  PAR FAIL: %s\n", conditionMessage(e))); NULL })
  } else results <- NULL
  if (is.null(results)) {
    cat("  Sequential fallback...\n")
    results <- lapply(seq_len(nrow(jobs)), function(i) {
      if (i %% 500 == 0) cat(sprintf("    %d/%d\n", i, nrow(jobs)))
      run_sl_fit(smooth_list[[jobs$sid[i]]], jobs$roi[i],
                 jobs$sid[i], jobs$session[i], jobs$r_scale[i])
    })
  }
  el <- (proc.time() - t0)["elapsed"]
  nok <- sum(vapply(results, function(r) is.list(r) && is.finite(r$a %||% NA), logical(1)))
  log_msg(sprintf("  %s: %d fits %.0fs OK=%d", label, length(results), el, nok))
  results
}

parse_res <- function(rl, grp) {
  do.call(rbind, lapply(rl, function(r) {
    if (!is.list(r)) return(NULL)
    data.frame(subject=r$subject %||% "", session=r$session %||% "",
      roi=r$roi %||% "", a=r$a %||% NA_real_,
      omega_hz=r$omega_hz %||% NA_real_, chisq=r$chisq %||% NA_real_,
      r_scale=r$r_scale %||% NA_real_, group=grp, stringsAsFactors=FALSE)
  }))
}

cat("Batch engine ready.\n")

Batch engine ready.


# 8 — Path 1: Multi-Session UKF (All HC Sessions + MDD r1/r2)

Run UKF on **all available HC sessions** (up to 10 per subject) plus both MDD
sessions, then average per-subject across sessions for reliable trait estimates.

In [8]:
log_section("PATH 1: MULTI-SESSION UKF")

build_jobs <- function(subjects, session_label, rscale_df) {
  jobs <- expand.grid(sid = subjects, roi = roi_cols, stringsAsFactors = FALSE)
  jobs$session <- session_label
  jobs$r_scale <- rscale_df$r_scale[match(jobs$sid, rscale_df$subject)]
  jobs
}

# ── HC: all sessions ─────────────────────────────────────────────
all_hc_results <- list()
for (s in hc_sessions) {
  s_chr <- as.character(s)
  subs_avail <- intersect(hc_subs, names(sm_hc[[s_chr]]))
  if (length(subs_avail) == 0) next
  rs_sub <- rscale_hc[rscale_hc$subject %in% subs_avail, ]
  jobs <- build_jobs(subs_avail, paste0("session_", s), rs_sub)
  res <- run_batch(sm_hc[[s_chr]], jobs, sprintf("HC s%d", s))
  all_hc_results[[s_chr]] <- parse_res(res, "HC")
}
hc_all_df <- do.call(rbind, all_hc_results)
hc_all_ok <- hc_all_df[is.finite(hc_all_df$a), ]
log_msg(sprintf("HC total valid fits: %d across %d sessions", nrow(hc_all_ok), length(hc_sessions)))

# ── MDD: both sessions ──────────────────────────────────────────
jobs_mr1 <- build_jobs(mdd_subs, "rest1", rscale_mdd)
jobs_mr2 <- build_jobs(mdd_subs, "rest2", rscale_mdd)
res_mr1 <- run_batch(sm_mdd_r1, jobs_mr1, "MDD r1")
res_mr2 <- run_batch(sm_mdd_r2, jobs_mr2, "MDD r2")
mdd_r1_df <- parse_res(res_mr1, "MDD"); mdd_r1_ok <- mdd_r1_df[is.finite(mdd_r1_df$a), ]
mdd_r2_df <- parse_res(res_mr2, "MDD"); mdd_r2_ok <- mdd_r2_df[is.finite(mdd_r2_df$a), ]
mdd_all_ok <- rbind(mdd_r1_ok, mdd_r2_ok)
log_msg(sprintf("MDD total valid: r1=%d r2=%d", nrow(mdd_r1_ok), nrow(mdd_r2_ok)))


PATH 1: MULTI-SESSION UKF
  HC s1: 6480 fits (16 cores)
[07:37:55]   HC s1: 6480 fits 5989s OK=6480 
  HC s2: 6480 fits (16 cores)
[09:17:46]   HC s2: 6480 fits 5988s OK=6480 
  HC s3: 6480 fits (16 cores)
[10:54:31]   HC s3: 6480 fits 5803s OK=6480 
  HC s4: 6264 fits (16 cores)
[12:23:59]   HC s4: 6264 fits 5366s OK=6264 
  HC s5: 6480 fits (16 cores)
[13:55:50]   HC s5: 6480 fits 5509s OK=6480 
  HC s6: 6264 fits (16 cores)
[15:19:13]   HC s6: 6264 fits 5000s OK=6264 
  HC s7: 6264 fits (16 cores)
[16:46:56]   HC s7: 6264 fits 5261s OK=6264 
  HC s8: 6264 fits (16 cores)
[18:17:06]   HC s8: 6264 fits 5408s OK=6264 
  HC s9: 6264 fits (16 cores)
[19:56:22]   HC s9: 6264 fits 5954s OK=6264 
  HC s10: 6480 fits (16 cores)
[21:46:07]   HC s10: 6480 fits 6583s OK=6480 
[21:46:08] HC total valid fits: 63720 across 10 sessions 
  MDD r1: 4104 fits (16 cores)
[22:44:23]   MDD r1: 4104 fits 3494s OK=4104 
  MDD r2: 4104 fits (16 cores)
[23:38:35]   MDD r2: 4104 fits 3251s OK=4104 
[23:38:37

# 9 — Multi-Session Averaged Estimates

Average $a$ per subject across all available sessions to obtain **reliable
trait-level estimates**. This is the key innovation over v3-final.

In [9]:
log_section("MULTI-SESSION AVERAGING")

# HC: average across all sessions per subject per ROI, then per subject
hc_avg_roi <- hc_all_ok |>
  group_by(subject, roi) |>
  summarise(a_mean = mean(a), n_sess = n(), .groups = "drop")

hc_avg_subj <- hc_avg_roi |>
  group_by(subject) |>
  summarise(mean_a = mean(a_mean), var_a = var(a_mean),
            n_rois = n(), n_sess_avg = mean(n_sess), .groups = "drop")

# MDD: average rest1 + rest2 per subject per ROI
mdd_avg_roi <- mdd_all_ok |>
  group_by(subject, roi) |>
  summarise(a_mean = mean(a), n_sess = n(), .groups = "drop")

mdd_avg_subj <- mdd_avg_roi |>
  group_by(subject) |>
  summarise(mean_a = mean(a_mean), var_a = var(a_mean),
            n_rois = n(), n_sess_avg = mean(n_sess), .groups = "drop")

log_msg(sprintf("HC  averaged: n=%d  mean_a=%.4f ± %.4f  avg_sessions=%.1f",
  nrow(hc_avg_subj), mean(hc_avg_subj$mean_a), sd(hc_avg_subj$mean_a),
  mean(hc_avg_subj$n_sess_avg)))
log_msg(sprintf("MDD averaged: n=%d  mean_a=%.4f ± %.4f  avg_sessions=%.1f",
  nrow(mdd_avg_subj), mean(mdd_avg_subj$mean_a), sd(mdd_avg_subj$mean_a),
  mean(mdd_avg_subj$n_sess_avg)))

# Variance reduction check
hc_s1_only <- hc_all_ok[hc_all_ok$session == "session_1", ] |>
  group_by(subject) |> summarise(a_s1 = mean(a), .groups = "drop")
log_msg(sprintf("HC SD reduction: single=%.4f → averaged=%.4f (%.1fx)",
  sd(hc_s1_only$a_s1), sd(hc_avg_subj$mean_a),
  sd(hc_s1_only$a_s1) / sd(hc_avg_subj$mean_a)))


MULTI-SESSION AVERAGING
[23:38:37] HC  averaged: n=30  mean_a=-0.2589 ± 0.0247  avg_sessions=9.8 
[23:38:37] MDD averaged: n=19  mean_a=-0.2666 ± 0.0790  avg_sessions=2.0 
[23:38:37] HC SD reduction: single=0.0695 → averaged=0.0247 (2.8x) 


# 10 — H1: Cross-Condition Test (Averaged Estimates)

In [10]:
log_section("H1 — AVERAGED ESTIMATES")

t_h1 <- t.test(hc_avg_subj$mean_a, mdd_avg_subj$mean_a)
d_h1 <- (mean(hc_avg_subj$mean_a) - mean(mdd_avg_subj$mean_a)) /
  sqrt((var(hc_avg_subj$mean_a) + var(mdd_avg_subj$mean_a)) / 2)
w_h1 <- wilcox.test(hc_avg_subj$mean_a, mdd_avg_subj$mean_a)

log_msg(sprintf("HC=%.4f±%.4f  MDD=%.4f±%.4f",
  mean(hc_avg_subj$mean_a), sd(hc_avg_subj$mean_a),
  mean(mdd_avg_subj$mean_a), sd(mdd_avg_subj$mean_a)))
log_msg(sprintf("Welch t=%.3f p=%.4e d=%.3f  MW W=%.0f p=%.4e",
  t_h1$statistic, t_h1$p.value, d_h1, w_h1$statistic, w_h1$p.value))

# TOST equivalence
pooled_sd <- sqrt((var(hc_avg_subj$mean_a) + var(mdd_avg_subj$mean_a)) / 2)
for (margin in c(0.5, 0.3)) {
  delta <- margin * pooled_sd
  diff_obs <- mean(hc_avg_subj$mean_a) - mean(mdd_avg_subj$mean_a)
  se <- sqrt(var(hc_avg_subj$mean_a)/nrow(hc_avg_subj) + var(mdd_avg_subj$mean_a)/nrow(mdd_avg_subj))
  t_lo <- (diff_obs + delta) / se; p_lo <- pt(t_lo, df = nrow(hc_avg_subj)+nrow(mdd_avg_subj)-2, lower.tail = FALSE)
  t_hi <- (diff_obs - delta) / se; p_hi <- pt(t_hi, df = nrow(hc_avg_subj)+nrow(mdd_avg_subj)-2, lower.tail = TRUE)
  p_tost <- max(p_lo, p_hi)
  log_msg(sprintf("TOST (d<%.1f, ±%.4f): p=%.4e equivalent=%s", margin, delta, p_tost, p_tost < 0.05))
}

# Bootstrap
boot_d <- replicate(10000, {
  bx <- sample(hc_avg_subj$mean_a, replace=TRUE); by <- sample(mdd_avg_subj$mean_a, replace=TRUE)
  (mean(bx)-mean(by))/sqrt((var(bx)+var(by))/2)
})
ci <- quantile(boot_d, c(0.025, 0.975))
log_msg(sprintf("Bootstrap d: %.3f  95%% CI [%.3f, %.3f]  contains 0: %s",
  d_h1, ci[1], ci[2], ci[1] <= 0 && ci[2] >= 0))


H1 — AVERAGED ESTIMATES
[23:38:37] HC=-0.2589±0.0247  MDD=-0.2666±0.0790 
[23:38:37] Welch t=0.416 p=6.8175e-01 d=0.133  MW W=399 p=1.8823e-02 
[23:38:37] TOST (d<0.5, ±0.0293): p=1.2782e-01 equivalent=FALSE 
[23:38:37] TOST (d<0.3, ±0.0176): p=3.0135e-01 equivalent=FALSE 
[23:38:37] Bootstrap d: 0.133  95% CI [-0.413, 1.446]  contains 0: TRUE 


# 11 — Multi-Session ICC

In [11]:
log_section("MULTI-SESSION ICC")

compute_icc <- function(s1, s2) {
  n <- length(s1); k <- 2; dat <- cbind(s1, s2)
  ms_r <- var(rowMeans(dat)) * k; ms_e <- mean(apply(dat, 1, var))
  ms_c <- var(colMeans(dat)) * n
  (ms_r - ms_e) / (ms_r + (k-1)*ms_e + k*(ms_c - ms_e)/n)
}

# HC: session 1 vs session-averaged (2-10)
hc_s1_sub <- hc_all_ok[hc_all_ok$session == "session_1", ] |>
  group_by(subject) |> summarise(a = mean(a), .groups = "drop")
hc_rest_sub <- hc_all_ok[hc_all_ok$session != "session_1", ] |>
  group_by(subject) |> summarise(a = mean(a), .groups = "drop")
hc_icc_df <- merge(hc_s1_sub, hc_rest_sub, by = "subject", suffixes = c("_s1", "_rest"))
if (nrow(hc_icc_df) >= 5) {
  icc_hc <- compute_icc(hc_icc_df$a_s1, hc_icc_df$a_rest)
  r_hc <- cor(hc_icc_df$a_s1, hc_icc_df$a_rest)
  log_msg(sprintf("HC ICC(s1 vs avg_rest): %.4f  r=%.4f", icc_hc, r_hc))
}

# HC: odd vs even sessions
hc_odd <- hc_all_ok[hc_all_ok$session %in% paste0("session_", seq(1,10,2)), ] |>
  group_by(subject) |> summarise(a = mean(a), .groups = "drop")
hc_even <- hc_all_ok[hc_all_ok$session %in% paste0("session_", seq(2,10,2)), ] |>
  group_by(subject) |> summarise(a = mean(a), .groups = "drop")
hc_oe <- merge(hc_odd, hc_even, by = "subject", suffixes = c("_odd", "_even"))
if (nrow(hc_oe) >= 5) {
  icc_hc_oe <- compute_icc(hc_oe$a_odd, hc_oe$a_even)
  r_hc_oe <- cor(hc_oe$a_odd, hc_oe$a_even)
  log_msg(sprintf("HC ICC(odd vs even): %.4f  r=%.4f (split-half reliability)", icc_hc_oe, r_hc_oe))
}

# MDD: rest1 vs rest2
mdd_s1 <- mdd_r1_ok |> group_by(subject) |> summarise(a = mean(a), .groups = "drop")
mdd_s2 <- mdd_r2_ok |> group_by(subject) |> summarise(a = mean(a), .groups = "drop")
mdd_icc_df <- merge(mdd_s1, mdd_s2, by = "subject", suffixes = c("_r1", "_r2"))
if (nrow(mdd_icc_df) >= 5) {
  icc_mdd <- compute_icc(mdd_icc_df$a_r1, mdd_icc_df$a_r2)
  r_mdd <- cor(mdd_icc_df$a_r1, mdd_icc_df$a_r2)
  log_msg(sprintf("MDD ICC(r1 vs r2): %.4f  r=%.4f", icc_mdd, r_mdd))
}


MULTI-SESSION ICC
[23:38:37] HC ICC(s1 vs avg_rest): 0.2708  r=0.4450 
[23:38:37] HC ICC(odd vs even): 0.2503  r=0.2933 (split-half reliability) 
[23:38:37] MDD ICC(r1 vs r2): 0.1531  r=0.2746 


# 12 — Path 3: Depression-Circuit Specific Analysis

Test $a$ in specific ROIs implicated in MDD: amygdala, hippocampus, mPFC,
dACC, NAcc, dlPFC, OFC, temporal pole. Uses **averaged** estimates.

In [12]:
log_section("PATH 3: DEPRESSION-CIRCUIT ANALYSIS")

# Identify depression-circuit ROIs
mdd_circuit_rois <- c()
for (roi in roi_cols) {
  # Subcortical
  if (roi %in% MDD_CIRCUIT_SUBCORT) { mdd_circuit_rois <- c(mdd_circuit_rois, roi); next }
  # Cortical patterns
  for (pat in MDD_CIRCUIT_CORTICAL_PATTERNS) {
    if (grepl(pat, roi)) { mdd_circuit_rois <- c(mdd_circuit_rois, roi); break }
  }
}
mdd_circuit_rois <- unique(mdd_circuit_rois)
log_msg(sprintf("Depression-circuit ROIs: %d out of %d", length(mdd_circuit_rois), N_ROIS))
for (r in mdd_circuit_rois) log_msg(sprintf("  %s", r))

# Averaged circuit-specific a per subject
hc_circuit <- hc_avg_roi[hc_avg_roi$roi %in% mdd_circuit_rois, ] |>
  group_by(subject) |>
  summarise(mean_a = mean(a_mean), .groups = "drop")

mdd_circuit <- mdd_avg_roi[mdd_avg_roi$roi %in% mdd_circuit_rois, ] |>
  group_by(subject) |>
  summarise(mean_a = mean(a_mean), .groups = "drop")

t_circ <- t.test(hc_circuit$mean_a, mdd_circuit$mean_a)
d_circ <- (mean(hc_circuit$mean_a) - mean(mdd_circuit$mean_a)) /
  sqrt((var(hc_circuit$mean_a) + var(mdd_circuit$mean_a)) / 2)
log_msg(sprintf("Circuit-specific: HC=%.4f±%.4f  MDD=%.4f±%.4f  d=%.3f p=%.4e",
  mean(hc_circuit$mean_a), sd(hc_circuit$mean_a),
  mean(mdd_circuit$mean_a), sd(mdd_circuit$mean_a),
  d_circ, t_circ$p.value))

# Per-ROI analysis within depression circuit
log_msg("\nPer-ROI depression circuit results:")
roi_results <- do.call(rbind, lapply(mdd_circuit_rois, function(roi) {
  hc_r <- hc_avg_roi[hc_avg_roi$roi == roi, ]
  mdd_r <- mdd_avg_roi[mdd_avg_roi$roi == roi, ]
  if (nrow(hc_r) < 5 || nrow(mdd_r) < 5) return(NULL)
  tt <- t.test(hc_r$a_mean, mdd_r$a_mean)
  d <- (mean(hc_r$a_mean) - mean(mdd_r$a_mean)) / sqrt((var(hc_r$a_mean) + var(mdd_r$a_mean)) / 2)
  data.frame(roi=roi, hc=mean(hc_r$a_mean), mdd=mean(mdd_r$a_mean), d=d, p=tt$p.value)
}))
if (!is.null(roi_results) && nrow(roi_results) > 0) {
  roi_results$p_fdr <- p.adjust(roi_results$p, method = "fdr")
  roi_results <- roi_results[order(roi_results$p), ]
  for (i in seq_len(nrow(roi_results))) {
    r <- roi_results[i, ]
    sig <- ifelse(r$p_fdr < 0.05, "**", ifelse(r$p < 0.05, "*", ""))
    log_msg(sprintf("  %-35s d=%+.3f p=%.4f p_fdr=%.4f %s", r$roi, r$d, r$p, r$p_fdr, sig))
  }
}


PATH 3: DEPRESSION-CIRCUIT ANALYSIS
[23:38:37] Depression-circuit ROIs: 69 out of 216 
[23:38:37]   7Networks_LH_SalVentAttn_Med_1 
[23:38:37]   7Networks_LH_SalVentAttn_Med_2 
[23:38:37]   7Networks_LH_SalVentAttn_Med_3 
[23:38:37]   7Networks_LH_Limbic_OFC_1 
[23:38:37]   7Networks_LH_Limbic_OFC_2 
[23:38:37]   7Networks_LH_Limbic_TempPole_1 
[23:38:37]   7Networks_LH_Limbic_TempPole_2 
[23:38:37]   7Networks_LH_Limbic_TempPole_3 
[23:38:37]   7Networks_LH_Limbic_TempPole_4 
[23:38:37]   7Networks_LH_Cont_PFCl_1 
[23:38:37]   7Networks_LH_Cont_PFCl_2 
[23:38:37]   7Networks_LH_Cont_PFCl_3 
[23:38:37]   7Networks_LH_Cont_PFCl_4 
[23:38:37]   7Networks_LH_Cont_PFCl_5 
[23:38:37]   7Networks_LH_Default_Temp_1 
[23:38:37]   7Networks_LH_Default_Temp_2 
[23:38:37]   7Networks_LH_Default_Temp_3 
[23:38:37]   7Networks_LH_Default_Temp_4 
[23:38:37]   7Networks_LH_Default_Temp_5 
[23:38:37]   7Networks_LH_Default_PFC_1 
[23:38:37]   7Networks_LH_Default_PFC_2 
[23:38:37]   7Networks_LH_Defa

# 13 — Path 2: Estimation Method Comparison

Three methods on the same data. This is the primary methodological contribution.

In [13]:
log_section("PATH 2: ESTIMATION METHOD COMPARISON")

# ── Method 1: UKF (already computed above) ───────────────────────
ukf_hc <- mean(hc_avg_subj$mean_a)
ukf_mdd <- mean(mdd_avg_subj$mean_a)

# ── Method 2: Spectral NLS Lorentzian ────────────────────────────
fit_lorentzian_nls <- function(ts, TR = 2.0, f_low = 0.01, f_high = 0.10) {
  ts <- ts[is.finite(ts)]; if (length(ts) < 30) return(NA_real_)
  ts <- ts - mean(ts); n <- length(ts)
  ft <- Mod(fft(ts))^2 / n; freqs <- seq(0, 1/(2*TR), length.out = floor(n/2)+1)
  psd <- ft[1:length(freqs)]; band <- freqs >= f_low & freqs <= f_high
  if (sum(band) < 5) return(NA_real_)
  fb <- freqs[band]; pb <- psd[band]; pb <- pb / max(pb)
  f0_init <- fb[which.max(pb)]
  tryCatch({
    fit <- nls(pb ~ amp / ((2*pi*(fb - f0))^2 + a_abs^2),
      start = list(amp = max(pb)*0.01, f0 = f0_init, a_abs = 2*pi*0.02),
      lower = list(amp = 1e-6, f0 = f_low, a_abs = 1e-4),
      upper = list(amp = 10, f0 = f_high, a_abs = 2.0),
      algorithm = "port", control = list(maxiter = 200, warnOnly = TRUE))
    -abs(coef(fit)["a_abs"])
  }, error = function(e) {
    hm <- max(pb)/2; ah <- which(pb >= hm)
    if (length(ah) < 2) return(NA_real_)
    -2 * pi * (fb[max(ah)] - fb[min(ah)]) / 2
  })
}

cat("Spectral NLS...\n")
spec_rois <- roi_cols[seq(1, N_ROIS, length.out = min(40, N_ROIS))]
deco_hc <- sapply(hc_subs, function(sid) {
  vals <- sapply(spec_rois, function(r) fit_lorentzian_nls(sm_hc[["1"]][[sid]][[r]]))
  mean(vals, na.rm = TRUE)
})
deco_mdd <- sapply(mdd_subs, function(sid) {
  vals <- sapply(spec_rois, function(r) fit_lorentzian_nls(sm_mdd_r1[[sid]][[r]]))
  mean(vals, na.rm = TRUE)
})
t_spec <- t.test(deco_hc, deco_mdd)
d_spec <- (mean(deco_hc,na.rm=T)-mean(deco_mdd,na.rm=T))/sqrt((var(deco_hc,na.rm=T)+var(deco_mdd,na.rm=T))/2)

# ── Method 3: FC-matching ────────────────────────────────────────
cat("FC-matching...\n")
fc_match <- function(data_list, subjects, rois, n_subj = 5) {
  subs <- subjects[1:min(n_subj, length(subjects))]
  n_r <- min(50, length(rois)); sel <- rois[seq(1, length(rois), length.out = n_r)]
  a_grid <- seq(-0.50, 0.10, by = 0.02)
  sapply(subs, function(sid) {
    mat <- as.matrix(data_list[[sid]][, sel]); mat <- mat[complete.cases(mat), ]
    if (nrow(mat) < 30) return(NA_real_)
    emp_fc <- cor(mat); emp_u <- emp_fc[upper.tri(emp_fc)]
    best_a <- NA; best_r <- -1
    for (a_val in a_grid) {
      n_tr <- nrow(mat); dt_sim <- 0.1; steps <- n_tr * round(TR/dt_sim)
      om <- runif(n_r, 0.03, 0.07) * 2 * pi; G <- 0.5
      x <- matrix(rnorm(n_r*2, 0, 0.01), ncol=2); bold <- matrix(NA, n_tr, n_r)
      for (t_s in 1:steps) {
        zsq <- x[,1]^2 + x[,2]^2
        cr <- G * (emp_fc[1:n_r,1:n_r] %*% x[,1] - x[,1]*colSums(abs(emp_fc[1:n_r,1:n_r])))
        ci <- G * (emp_fc[1:n_r,1:n_r] %*% x[,2] - x[,2]*colSums(abs(emp_fc[1:n_r,1:n_r])))
        x[,1] <- x[,1] + (a_val*x[,1]-om*x[,2]-zsq*x[,1])*dt_sim + cr*dt_sim + rnorm(n_r,0,sqrt(dt_sim)*0.02)
        x[,2] <- x[,2] + (a_val*x[,2]+om*x[,1]-zsq*x[,2])*dt_sim + ci*dt_sim + rnorm(n_r,0,sqrt(dt_sim)*0.02)
        if (t_s %% round(TR/dt_sim) == 0) {
          idx <- t_s / round(TR/dt_sim)
          if (idx <= n_tr) bold[idx, ] <- x[,1]
        }
      }
      sim_fc <- tryCatch(cor(bold, use="pairwise.complete.obs"), error=function(e) NULL)
      if (is.null(sim_fc)) next
      sim_u <- sim_fc[upper.tri(sim_fc)]; v <- is.finite(emp_u) & is.finite(sim_u)
      if (sum(v) < 10) next
      r_val <- cor(emp_u[v], sim_u[v])
      if (!is.na(r_val) && r_val > best_r) { best_r <- r_val; best_a <- a_val }
    }
    best_a
  })
}
fcm_hc <- fc_match(sm_hc[["1"]], hc_subs, roi_cols)
fcm_mdd <- fc_match(sm_mdd_r1, mdd_subs, roi_cols)

# ── Summary table ────────────────────────────────────────────────
log_msg("\n=== ESTIMATION METHOD COMPARISON ===")
log_msg(sprintf("  UKF (per-subj, averaged): HC=%.4f  MDD=%.4f  d=%.3f p=%.4e",
  ukf_hc, ukf_mdd, d_h1, t_h1$p.value))
log_msg(sprintf("  Spectral NLS:            HC=%.4f  MDD=%.4f  d=%.3f p=%.4e",
  mean(deco_hc,na.rm=T), mean(deco_mdd,na.rm=T), d_spec, t_spec$p.value))
log_msg(sprintf("  FC-matching:             HC=%.4f  MDD=%.4f",
  mean(fcm_hc,na.rm=T), mean(fcm_mdd,na.rm=T)))
log_msg(sprintf("  Deco 2017 (published):   ~-0.02"))
log_msg(sprintf("  Ratio UKF/Spectral: %.1fx", ukf_hc / mean(deco_hc, na.rm=T)))


PATH 2: ESTIMATION METHOD COMPARISON
Spectral NLS...


Warning message in nls(pb ~ amp/((2 * pi * (fb - f0))^2 + a_abs^2), start = list(amp = max(pb) * :
“Convergence failure: false convergence (8)”
Warning message in nls(pb ~ amp/((2 * pi * (fb - f0))^2 + a_abs^2), start = list(amp = max(pb) * :
“Convergence failure: false convergence (8)”
Warning message in nls(pb ~ amp/((2 * pi * (fb - f0))^2 + a_abs^2), start = list(amp = max(pb) * :
“Convergence failure: false convergence (8)”
Warning message in nls(pb ~ amp/((2 * pi * (fb - f0))^2 + a_abs^2), start = list(amp = max(pb) * :
“Convergence failure: function evaluation limit reached without convergence (9)”
Warning message in nls(pb ~ amp/((2 * pi * (fb - f0))^2 + a_abs^2), start = list(amp = max(pb) * :
“Convergence failure: false convergence (8)”
Warning message in nls(pb ~ amp/((2 * pi * (fb - f0))^2 + a_abs^2), start = list(amp = max(pb) * :
“Convergence failure: false convergence (8)”
Warning message in nls(pb ~ amp/((2 * pi * (fb - f0))^2 + a_abs^2), start = list(amp = max(pb) * :
“

FC-matching...
[23:39:43] 
=== ESTIMATION METHOD COMPARISON === 
[23:39:43]   UKF (per-subj, averaged): HC=-0.2589  MDD=-0.2666  d=0.133 p=6.8175e-01 
[23:39:43]   Spectral NLS:            HC=-0.1576  MDD=-0.1500  d=-0.144 p=6.4053e-01 
[23:39:43]   FC-matching:             HC=0.0120  MDD=0.0000 
[23:39:43]   Deco 2017 (published):   ~-0.02 
[23:39:43]   Ratio UKF/Spectral: 1.6x 


# 14 — Comprehensive Summary

In [14]:
log_section("COMPREHENSIVE SUMMARY")

log_msg("=== H1: CROSS-CONDITION (averaged) ===")
log_msg(sprintf("  d=%.3f  p=%.4e", d_h1, t_h1$p.value))
log_msg(sprintf("  Bootstrap 95%% CI: [%.3f, %.3f]", ci[1], ci[2]))

log_msg("\n=== ICC (multi-session) ===")
if (exists("icc_hc_oe")) log_msg(sprintf("  HC split-half (odd/even): %.4f", icc_hc_oe))
if (exists("icc_hc")) log_msg(sprintf("  HC s1 vs rest: %.4f", icc_hc))
if (exists("icc_mdd")) log_msg(sprintf("  MDD r1 vs r2: %.4f", icc_mdd))

log_msg("\n=== DEPRESSION CIRCUIT ===")
log_msg(sprintf("  Circuit-specific d=%.3f p=%.4e", d_circ, t_circ$p.value))
if (!is.null(roi_results) && nrow(roi_results) > 0) {
  top <- head(roi_results[order(roi_results$p), ], 5)
  for (i in seq_len(nrow(top))) {
    r <- top[i, ]
    log_msg(sprintf("  Top ROI: %-30s d=%+.3f p=%.4f", r$roi, r$d, r$p))
  }
}

log_msg("\n=== METHOD COMPARISON ===")
log_msg(sprintf("  UKF:      %.4f (HC)  %.4f (MDD)", ukf_hc, ukf_mdd))
log_msg(sprintf("  Spectral: %.4f (HC)  %.4f (MDD)", mean(deco_hc,na.rm=T), mean(deco_mdd,na.rm=T)))
log_msg(sprintf("  FC-match: %.4f (HC)  %.4f (MDD)", mean(fcm_hc,na.rm=T), mean(fcm_mdd,na.rm=T)))

log_msg("\n=== VARIANCE REDUCTION ===")
log_msg(sprintf("  HC SD: single=%.4f → averaged=%.4f",
  sd(hc_s1_only$a_s1), sd(hc_avg_subj$mean_a)))

saveRDS(list(
  hc_all=hc_all_ok, mdd_r1=mdd_r1_ok, mdd_r2=mdd_r2_ok,
  hc_avg=hc_avg_subj, mdd_avg=mdd_avg_subj,
  hc_avg_roi=hc_avg_roi, mdd_avg_roi=mdd_avg_roi,
  rscale=list(hc=rscale_hc, mdd=rscale_mdd),
  circuit_results=roi_results,
  method_comparison=data.frame(
    method=c("UKF","Spectral","FC-match"),
    hc=c(ukf_hc, mean(deco_hc,na.rm=T), mean(fcm_hc,na.rm=T)),
    mdd=c(ukf_mdd, mean(deco_mdd,na.rm=T), mean(fcm_mdd,na.rm=T)))
), file.path(RESULTS_DIR, "ch5_v4def_results.rds"))

log_msg(sprintf("\nSaved: %s", file.path(RESULTS_DIR, "ch5_v4def_results.rds")))
log_section(sprintf("CH5 v4-DEFINITIVE COMPLETE — %s", Sys.time()))
close(log_con)
cat(sprintf("\nLog: %s\n", LOG_FILE))


COMPREHENSIVE SUMMARY
[23:39:43] === H1: CROSS-CONDITION (averaged) === 
[23:39:43]   d=0.133  p=6.8175e-01 
[23:39:43]   Bootstrap 95% CI: [-0.413, 1.446] 
[23:39:43] 
=== ICC (multi-session) === 
[23:39:43]   HC split-half (odd/even): 0.2503 
[23:39:43]   HC s1 vs rest: 0.2708 
[23:39:43]   MDD r1 vs r2: 0.1531 
[23:39:43] 
=== DEPRESSION CIRCUIT === 
[23:39:43]   Circuit-specific d=0.169 p=5.9801e-01 
[23:39:43]   Top ROI: 7Networks_RH_Default_Temp_2    d=+0.937 p=0.0058 
[23:39:43]   Top ROI: 7Networks_LH_Default_PFC_7     d=+0.780 p=0.0131 
[23:39:43]   Top ROI: 7Networks_LH_Limbic_TempPole_1 d=+0.669 p=0.0277 
[23:39:43]   Top ROI: 7Networks_RH_Default_Temp_4    d=+0.715 p=0.0280 
[23:39:43]   Top ROI: 7Networks_LH_Default_PFC_2     d=+0.717 p=0.0285 
[23:39:43] 
=== METHOD COMPARISON === 
[23:39:43]   UKF:      -0.2589 (HC)  -0.2666 (MDD) 
[23:39:43]   Spectral: -0.1576 (HC)  -0.1500 (MDD) 
[23:39:43]   FC-match: 0.0120 (HC)  0.0000 (MDD) 
[23:39:43] 
=== VARIANCE REDUCTION ===